# Agent 4: Text Cleaning and NLP Analyzer
Responsible for extracting, cleaning, and analyzing open-ended text fields (Feedback, Motivation, General Comments) with strict local privacy.

In [1]:
# 1. Unzip dataset and setup directories
import zipfile
from pathlib import Path
import pandas as pd

zip_path = Path("data.zip")
extract_dir = Path("extracted_data")
output_dir = Path("cleaned_data")

extract_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    print("data.zip extracted successfully.")
else:
    print("data.zip not found in current directory.")

excel_files = sorted(list(extract_dir.rglob("*.xlsx")))
print(f"Total Excel files found: {len(excel_files)}")

data.zip extracted successfully.
Total Excel files found: 12


In [2]:
# 2. Agent 4 Core Logic: Privacy Filter, Cleanup & Local NLP
import re
from textblob import TextBlob

POSITIVE_WORDS_AR = {
    "ممتاز", "رائع", "مفيد", "جميل", "استفدت", "شكرا", "شكراً", "قيمة", 
    "تطوير", "مهارات", "فرصة", "اهتمام", "متحمس", "مبدع", "عظيم", "جيد", "التعرف", "ريادة"
}

NEGATIVE_WORDS_AR = {
    "سيء", "سيئ", "صعب", "ممل", "تشتت", "تضييع", "مشكلة", 
    "ضعيف", "تحديات", "عقبات", "غير مفيد", "تأخير", "ازعاج"
}

STOPWORDS = {
    "من", "الى", "إلى", "عن", "على", "في", "كل", "مع", "هذا", "هذه", "التي", "الذي", "اللي", "كيفية",
    "and", "the", "in", "to", "for", "with", "is", "of", "it", "my", "i", "a", "an", "am", "very"
}

def mask_pii_locally(text: str) -> str:
    """Local Privacy Guard: Redact email, phone, and national ID."""
    if not text:
        return text
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    text = re.sub(email_pattern, '[EMAIL_REDACTED]', text)
    phone_pattern = r'(?:\+?966|0)?5\d{8}|\+?\d{10,14}'
    text = re.sub(phone_pattern, '[PHONE_REDACTED]', text)
    id_pattern = r'\b\d{10,}\b'
    text = re.sub(id_pattern, '[ID_REDACTED]', text)
    return text

def clean_text_basic(text):
    """Whitespace trimming, dummy response removal, and local privacy masking."""
    if pd.isna(text):
        return None
    val = str(text).strip()
    val = re.sub(r'\s+', ' ', val)
    val = mask_pii_locally(val)
    empty_patterns = {
        "-", ".", "لا يوجد", "لايوجد", "مافي", "none", "nan", "null", "no", "لاشيء", "n/a", "na"
    }
    if val.lower() in empty_patterns or len(val) < 2:
        return None
    return val

def extract_local_topics(text: str) -> str:
    """Extract up to 3 distinct significant keywords locally."""
    words = re.findall(r'\b\w{3,}\b', str(text).lower())
    keywords = [w for w in words if w not in STOPWORDS]
    top_keys = list(dict.fromkeys(keywords))[:3]
    return ", ".join(top_keys) if top_keys else "general"

def analyze_sentiment_locally(text: str) -> str:
    """Rule-based Arabic sentiment classification and TextBlob polarity for English."""
    text_str = str(text)
    ar_pos = sum(1 for w in POSITIVE_WORDS_AR if w in text_str)
    ar_neg = sum(1 for w in NEGATIVE_WORDS_AR if w in text_str)

    if ar_pos > ar_neg:
        return "Positive"
    elif ar_neg > ar_pos:
        return "Negative"
    elif ar_pos > 0 and ar_pos == ar_neg:
        return "Mixed"

    try:
        polarity = TextBlob(text_str).sentiment.polarity
        if polarity > 0.05:
            return "Positive"
        elif polarity < -0.05:
            return "Negative"
        else:
            return "Neutral"
    except Exception:
        return "Neutral"

def clean_and_analyze_feedback(text):
    """Local pipeline unifying cleanup, privacy, sentiment, and topic extraction."""
    cleaned = clean_text_basic(text)
    if cleaned is None:
        return {
            "cleaned_text": "",
            "sentiment": "None",
            "topics": ""
        }
    return {
        "cleaned_text": cleaned,
        "sentiment": analyze_sentiment_locally(cleaned),
        "topics": extract_local_topics(cleaned)
    }

def process_dataframe(df, target_columns):
    """Appends enriched columns to DataFrame without mutating original raw columns."""
    df_out = df.copy()
    for col in target_columns:
        if col not in df_out.columns:
            continue
        cleaned_list = []
        sentiment_list = []
        topics_list = []
        for val in df_out[col]:
            res = clean_and_analyze_feedback(val)
            cleaned_list.append(res["cleaned_text"])
            sentiment_list.append(res["sentiment"])
            topics_list.append(res["topics"])
        df_out[f"{col}_cleaned"] = cleaned_list
        df_out[f"{col}_sentiment"] = sentiment_list
        df_out[f"{col}_topics"] = topics_list
    return df_out

In [3]:
# 3. Execute Processing on all 12 Files and Save Cleaned Data
keywords = ["motivated", "ملاحظات", "تقييم", "اقتراح", "feedback", "سبب", "اذكر", "what", "how"]

for file_path in excel_files:
    df = pd.read_excel(file_path)
    target_cols = [
        col for col in df.columns 
        if any(kw in str(col).lower() for kw in keywords)
    ]
    if target_cols:
        print(f"Processing: {file_path.name} -> Columns: {target_cols}")
        cleaned_df = process_dataframe(df, target_columns=target_cols)
    else:
        cleaned_df = df
    save_path = output_dir / f"Cleaned_{file_path.name}"
    cleaned_df.to_excel(save_path, index=False)

print("\n--- Pipeline Execution Completed Locally with 100% Privacy! ---")

Processing: Entrepreneurship Workshop Registration Form (الحضور).xlsx -> Columns: ['?What motivated you to register for this workshop']
Processing: الحوكمة الرقمية (Responses) (3).xlsx -> Columns: ['What do you expect to gain from this workshop on digital governance workshop?', 'What challenges do you face when applying digital governance principles in your work or studies?']
Processing: تسجيل الحوكمة الرقمية (1).xlsx -> Columns: ['What do you expect to gain from this workshop on digital governance workshop?', 'What challenges do you face when applying digital governance principles in your work or studies?']
Processing: تسجيل معسكر AI (1).xlsx -> Columns: ['How often do you use them?  ', 'What do you use them for?  ']
Processing: تسجيل ورشة عصر التشتت.xlsx -> Columns: ['What do you expect to gain from this workshop on distraction?', '\nWhat is your biggest source of distraction?']
Processing: نموذج طلب الانضمام إلى لجنة المطورين لعام 2026 (Responses).xlsx -> Columns: ['قيم مدى ارتياحك 

In [4]:
# 4. Inspection & Verification
cleaned_files = sorted(list(output_dir.glob("Cleaned_*.xlsx")))
target_file = None
target_df = None
cleaned_cols = []

for file_path in cleaned_files:
    df = pd.read_excel(file_path)
    cols = [c for c in df.columns if c.endswith("_cleaned")]
    if cols:
        target_file = file_path
        target_df = df
        cleaned_cols = cols
        break

if target_file is not None:
    base = cleaned_cols[0].replace("_cleaned", "")
    print(f"Verified File: {target_file.name}")
    print("--- Sentiment Distribution ---")
    print(target_df[f"{base}_sentiment"].value_counts())
    print("\n--- Sample Cleaned Records (First 5 Non-Empty Rows) ---")
    valid_rows = target_df[target_df[f"{base}_cleaned"].notna() & (target_df[f"{base}_cleaned"] != "")].head(5)
    for idx, row in valid_rows.iterrows():
        print(f"Cleaned Text : {row[f'{base}_cleaned']}")
        print(f"Sentiment    : {row[f'{base}_sentiment']}")
        print(f"Topics       : {row[f'{base}_topics']}")
        print("-" * 50)
else:
    print("No processed columns found.")

Verified File: Cleaned_Entrepreneurship Workshop Registration Form (الحضور).xlsx
--- Sentiment Distribution ---
?What motivated you to register for this workshop_sentiment
Positive    35
Neutral     30
Negative     2
Mixed        1
Name: count, dtype: int64

--- Sample Cleaned Records (First 5 Non-Empty Rows) ---
Cleaned Text : Because it focuses on entrepreneurship, and I am very interested in this field and everything related to business and companies.
Sentiment    : Positive
Topics       : because, focuses, entrepreneurship
--------------------------------------------------
Cleaned Text : My willingness to learn from other’s experiences
Sentiment    : Negative
Topics       : willingness, learn, from
--------------------------------------------------
Cleaned Text : التعرف على كيفية ريادة الاعمال والتحديات والعقبات اللي ممكن تواجهني
Sentiment    : Mixed
Topics       : التعرف, ريادة, الاعمال
--------------------------------------------------
Cleaned Text : to gain practical skills expa